In [1]:
import logging
import shutil
import socket
import sys
from datetime import UTC, datetime
from math import prod
from pathlib import Path
from pprint import pformat
from time import perf_counter

import numpy as np
from memory_tempfile import MemoryTempfile as MemoryTemper
from tifffile import TiffFile

from comptest import imagemagick, libtiff, tifffile
from comptest.constants import DATASETS_DIR
from comptest.types import (
    DTYPE_ANY,
    DTYPE_BOOL,
    DTYPE_FLOAT,
    DTYPE_UINT8,
    DTYPE_UINT8_UINT16,
    DTYPE_UINT32,
    CompressionTest,
    FileStackCompressionConfig,
)

%load_ext autoreload
%autoreload 2

# Parameters

In [2]:
reverse_configs: bool = False
target_configs: str | None = None
is_papermill: bool = False

# Setup

In [3]:
def configure_logging(log_dir: Path) -> tuple[logging.Logger, Path]:
    log_file = log_dir / "compression.log"

    configured_logger = logging.getLogger(__name__)
    configured_logger.setLevel(logging.DEBUG)
    configured_logger.propagate = False

    # Avoid duplicate handlers when notebook cells are rerun.
    for handler in list(configured_logger.handlers):
        if getattr(handler, "_compression_notebook_handler", False):
            configured_logger.removeHandler(handler)
            handler.close()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    stdout_handler = logging.StreamHandler(sys.stdout)
    stdout_handler.setLevel(logging.DEBUG)
    stdout_handler.setFormatter(formatter)
    stdout_handler._compression_notebook_handler = True

    file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)
    file_handler._compression_notebook_handler = True

    configured_logger.addHandler(stdout_handler)
    configured_logger.addHandler(file_handler)

    return configured_logger, log_file

In [4]:
PROJECT_DIR = Path().resolve()
OUTPUT_DIR = PROJECT_DIR / "output"
assert OUTPUT_DIR.is_dir()

mem_temper = MemoryTemper(filesystem_types=["tmpfs"])

In [5]:
COMPRESSION_TESTS_DIR = Path(
    "/dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests"
)
assert COMPRESSION_TESTS_DIR.is_dir()

In [6]:
DATASET = "Canislupus_CZM_712"
DATASET_DIR = DATASETS_DIR / DATASET
assert DATASET_DIR.is_dir()

SCRATCH_INPUT_FILES = list(DATASET_DIR.glob("*.tif"))

In [7]:
# Load input files into memory
# !!! Note: We currently don't clean this up after use!
temp_dir_path = (
    Path(mem_temper.gettempdir()) / "s250250-compression-tests-filestack-in-memory"
)
print(temp_dir_path)

if temp_dir_path.exists():
    shutil.rmtree(temp_dir_path)

temp_dir_path.mkdir(parents=True, exist_ok=True)
INPUT_FILES = []
for file in SCRATCH_INPUT_FILES:
    shutil.copy(file, temp_dir_path / file.name)
    INPUT_FILES.append(temp_dir_path / file.name)


/dev/shm/s250250-compression-tests-filestack-in-memory


In [8]:
# Check dtype is the same for all
INPUT_DTYPE = None
for file in INPUT_FILES:
    with TiffFile(file) as tif:
        dtype = tif.pages[0].dtype
        if INPUT_DTYPE is None:
            INPUT_DTYPE = dtype
        else:
            assert dtype == INPUT_DTYPE, (
                f"Input files have different dtypes: {dtype} != {INPUT_DTYPE}"
            )
INPUT_DTYPE

dtype('uint16')

# Run

In [ ]:
host = socket.gethostname()
TEST_CATEGORY = "full"
TEST_HOST = host
TEST_ID = "none"
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

TEST_OUTPUT_DIR = (
    OUTPUT_DIR
    / "file-stack"
    / f"{TEST_CATEGORY}_{RUN_TIMESTAMP}_{TEST_HOST}_{TEST_ID}"
    / DATASET
)
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger, LOG_FILE = configure_logging(TEST_OUTPUT_DIR)

logger.info("Initialized compression test run")
logger.info("Host: %s", host)
logger.info("Dataset: %s", DATASET)
logger.info("Test output directory: %s", TEST_OUTPUT_DIR)
logger.info("Log file: %s", LOG_FILE)

2026-08-02 14:08:30 | INFO     | Initialized compression test run
2026-08-02 14:08:30 | INFO     | Host: n-62-11-19
2026-08-02 14:08:30 | INFO     | Dataset: Canislupus_CZM_712
2026-08-02 14:08:30 | INFO     | Test output directory: /zhome/1a/a/219376/Code/qim-naturarv3d-compression/output/file-stack/full_n-62-11-19_2026-08-02_14-08-30/Canislupus_CZM_712
2026-08-02 14:08:30 | INFO     | Log file: /zhome/1a/a/219376/Code/qim-naturarv3d-compression/output/file-stack/full_n-62-11-19_2026-08-02_14-08-30/Canislupus_CZM_712/compression.log


In [10]:
def _lt_comp(*args: str):
    return (libtiff.compress_bulk, args, {})


def _im_comp(*args: str):
    return (imagemagick.compress_bulk, args, {})


def _tf_comp(compression: str, **kwargs):
    return (tifffile.compress_bulk, (compression,), kwargs)


_C = FileStackCompressionConfig

_BC = {"paradigm": "file_stack"}
_BC_LIBTIFF = _BC | {
    "implementation": "libtiff/tiffcp",
    "decompressor": (libtiff.decompress_bulk, (), {}),
    "comparer": (libtiff.compare_bulk, ("-t",), {}),
}
_BC_IMAGEMAGICK = _BC | {
    "implementation": "imagemagick",
    "decompressor": (imagemagick.decompress_bulk, (), {}),
    "comparer": (imagemagick.compare_bulk, (), {}),
}
_BC_TIFFFILE = _BC | {
    "implementation": "tifffile",
    "decompressor": (tifffile.decompress_bulk, (), {}),
    "comparer": (tifffile.compare_bulk, (), {}),
}
_BC_LOSSLESS_LIBTIFF = _BC_LIBTIFF | {"lossless": True}
_BC_LOSSLESS_IMAGEMAGICK = _BC_IMAGEMAGICK | {"lossless": True}
_BC_LOSSLESS_TIFFFILE = _BC_TIFFFILE | {"lossless": True}

_PREDICTORS = {1: "none", 2: "horizontal", 3: "float"}
_DEFLATE_EFFORTS = [1, 2, 3, 4, 5, 7, 9, 12]
_LZMA_EFFORTS = [1, 2, 3, 4, 5, 7, 9]
_ZSTD_EFFORTS = [1, 5, 9, 13, 17, 19, 22]
_PNG_EFFORTS = [1, 2, 3, 4, 5, 7, 9]
_JPEG_1_QUALITIES = [25, 50, 75, 90, 100]
_JPEG_XR_QUALITIES = [25, 50, 75, 90, 100]
_JPEG2000_QUALITIES = [25, 50, 75, 90, 100]

BASELINE_CONFIGS = [
    _C(
        name="libtiff_baseline",
        series="tiff-baseline",
        allowed_dtypes=DTYPE_ANY,
        compressor=_lt_comp("-c", "none"),
        **_BC_LOSSLESS_LIBTIFF,
    ),
    _C(
        name="imagemagick_baseline",
        series="tiff-baseline",
        allowed_dtypes=DTYPE_ANY,
        compressor=_im_comp("-compress", "None"),
        **_BC_LOSSLESS_IMAGEMAGICK,
    ),
    _C(
        name="tifffile_baseline",
        series="tiff-baseline",
        allowed_dtypes=DTYPE_ANY,
        compressor=(tifffile.compress_bulk, (), {}),
        **_BC_LOSSLESS_TIFFFILE,
    ),
]

# --- LZW ---
LIBTIFF_LZW_CONFIGS = [
    _C(
        name=f"libtiff_lzw_pred{pred_idx}",
        series="tiff-lzw",
        differencing=pred_name,
        predictor=pred_idx,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=_lt_comp("-c", f"lzw:{pred_idx}"),
        **_BC_LOSSLESS_LIBTIFF,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
]
IMAGEMAGICK_LZW_CONFIGS = [
    _C(
        name=f"imagemagick_lzw_pred{pred_idx}",
        series="tiff-lzw",
        differencing=pred_name,
        predictor=pred_idx,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=_im_comp(
            "-compress", "LZW", "-define", f"tiff:predictor={pred_idx}"
        ),
        **_BC_LOSSLESS_IMAGEMAGICK,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
]
TIFFFILE_LZW_CONFIGS = [
    _C(
        name=f"tifffile_lzw_pred{pred_idx}",
        series="tiff-lzw",
        differencing=pred_name,
        predictor=pred_idx,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=(tifffile.compress_bulk, ("lzw",), {"predictor": pred_idx}),
        **_BC_LOSSLESS_TIFFFILE,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
]
LZW_CONFIGS = LIBTIFF_LZW_CONFIGS + IMAGEMAGICK_LZW_CONFIGS + TIFFFILE_LZW_CONFIGS

# --- DEFLATE ---
LIBTIFF_DEFLATE_CONFIGS = [
    _C(
        name=f"libtiff_deflate_pred{pred_idx}_s{subcodec_idx}_p{effort}",
        series="tiff-deflate",
        differencing=pred_name,
        predictor=pred_idx,
        effort=effort,
        subcodec=subcodec_name,
        subcodec_index=subcodec_idx,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=_lt_comp("-c", f"zip:{pred_idx}:s{subcodec_idx}:p{effort}"),
        **_BC_LOSSLESS_LIBTIFF,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
    for effort in _DEFLATE_EFFORTS
    for subcodec_idx, subcodec_name in {0: "zlib", 1: "libdeflate"}.items()
]

IMAGEMAGICK_DEFLATE_CONFIGS = [
    _C(
        name=f"imagemagick_deflate_pred{pred_idx}_p{effort}",
        series="tiff-deflate",
        differencing=pred_name,
        predictor=pred_idx,
        effort=effort,
        subcodec=None,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=_im_comp(
            "-compress",
            "Zip",
            "-define",
            f"tiff:predictor={pred_idx}",
            "-quality",
            f"{effort * 10}",
        ),
        **_BC_LOSSLESS_IMAGEMAGICK,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
    for effort in _DEFLATE_EFFORTS
    # Note: No subcodec available in ImageMagick, so no s=0/1 loop here
]

TIFFFILE_DEFLATE_CONFIGS = [
    _C(
        name=f"tifffile_deflate_pred{pred_idx}_p{effort}",
        series="tiff-deflate",
        differencing=pred_name,
        predictor=pred_idx,
        effort=effort,
        subcodec=None,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=(
            tifffile.compress_bulk,
            ("zlib",),
            {"compressionargs": {"level": effort}, "predictor": pred_idx},
        ),
        **_BC_LOSSLESS_TIFFFILE,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
    for effort in _DEFLATE_EFFORTS
]

DEFLATE_CONFIGS = (
    LIBTIFF_DEFLATE_CONFIGS + IMAGEMAGICK_DEFLATE_CONFIGS + TIFFFILE_DEFLATE_CONFIGS
)

# --- LZMA ---
LIBTIFF_LZMA_CONFIGS = [
    _C(
        name=f"libtiff_lzma_pred{pred_idx}_p{effort}",
        series="tiff-lzma",
        differencing=pred_name,
        predictor=pred_idx,
        effort=effort,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=_lt_comp("-c", f"lzma:{pred_idx}:p{effort}"),
        **_BC_LOSSLESS_LIBTIFF,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
    for effort in _LZMA_EFFORTS
]
IMAGEMAGICK_LZMA_CONFIGS = [
    _C(
        name=f"imagemagick_lzma_pred{pred_idx}_p{effort}",
        series="tiff-lzma",
        differencing=pred_name,
        predictor=pred_idx,
        effort=effort,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=_im_comp(
            "-compress",
            "LZMA",
            "-define",
            f"tiff:predictor={pred_idx}",
            "-quality",
            f"{effort * 10}",
        ),
        **_BC_LOSSLESS_IMAGEMAGICK,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
    for effort in _LZMA_EFFORTS
]
TIFFFILE_LZMA_CONFIGS = [
    _C(
        name=f"tifffile_lzma_pred{pred_idx}_p{effort}",
        series="tiff-lzma",
        differencing=pred_name,
        predictor=pred_idx,
        effort=effort,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=(
            tifffile.compress_bulk,
            ("lzma",),
            {"compressionargs": {"level": effort}, "predictor": pred_idx},
        ),
        **_BC_LOSSLESS_TIFFFILE,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
    for effort in _LZMA_EFFORTS
]
LZMA_CONFIGS = LIBTIFF_LZMA_CONFIGS + IMAGEMAGICK_LZMA_CONFIGS + TIFFFILE_LZMA_CONFIGS

# --- Zstd ---
LIBTIFF_ZSTD_CONFIGS = [
    _C(
        name=f"libtiff_zstd_pred{pred_idx}_p{effort}",
        series="tiff-zstd",
        differencing=pred_name,
        predictor=pred_idx,
        effort=effort,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=_lt_comp("-c", f"zstd:{pred_idx}:p{effort}"),
        **_BC_LOSSLESS_LIBTIFF,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
    for effort in _ZSTD_EFFORTS
]

IMAGEMAGICK_ZSTD_CONFIGS = [
    _C(
        name=f"imagemagick_zstd_pred{pred_idx}_p{effort}",
        series="tiff-zstd",
        differencing=pred_name,
        predictor=pred_idx,
        effort=effort,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=_im_comp(
            "-compress",
            "Zstd",
            "-define",
            f"tiff:predictor={pred_idx}",
            "-quality",
            f"{effort * 5}",  # Roughly!
        ),
        **_BC_LOSSLESS_IMAGEMAGICK,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
    for effort in _ZSTD_EFFORTS
]
TIFFFILE_ZSTD_CONFIGS = [
    _C(
        name=f"tifffile_zstd_pred{pred_idx}_p{effort}",
        series="tiff-zstd",
        differencing=pred_name,
        predictor=pred_idx,
        effort=effort,
        allowed_dtypes=DTYPE_FLOAT if pred_idx == 3 else DTYPE_ANY,
        compressor=(
            tifffile.compress_bulk,
            ("zstd",),
            {"compressionargs": {"level": effort}, "predictor": pred_idx},
        ),
        **_BC_LOSSLESS_TIFFFILE,
    )
    for pred_idx, pred_name in _PREDICTORS.items()
    for effort in _ZSTD_EFFORTS
]
ZSTD_CONFIGS = LIBTIFF_ZSTD_CONFIGS + IMAGEMAGICK_ZSTD_CONFIGS + TIFFFILE_ZSTD_CONFIGS

# --- LERC ---
_LIBTIFF_LERC_SUBCODECS = {
    0: "none",
    1: "deflate",
    2: "zstd",
}

_LERC_COMBINATIONS = []
for p in [1, 5, 8, 10, 12, 14, 16, 18, 20, 22]:
    for s in [0, 1, 2]:
        if s == 1 and p > 12:
            continue
        _LERC_COMBINATIONS.append({"z": 0, "s": s, "p": p})

LIBTIFF_LERC_CONFIGS = [
    _C(
        name=f"libtiff_lerc_effort_z{c['z']}_s{c['s']}_p{c['p']}",
        series="tiff-lerc",
        subseries="tiff-lerc-libtiff",
        effort=c["p"],
        subcodec=_LIBTIFF_LERC_SUBCODECS[c["s"]],
        subcodec_index=c["s"],
        z_max_error=c["z"],
        allowed_dtypes=DTYPE_ANY,
        compressor=_lt_comp("-c", f"lerc:{c['z']}:s{c['s']}:p{c['p']}"),
        **_BC_LOSSLESS_LIBTIFF,
    )
    for c in _LERC_COMBINATIONS
]


def _resolve_imagemagick_lerc(z, s, p):
    out = [
        "-compress",
        "LERC",
        "-define",
        f"tiff:lerc-maxzerror={z}",
    ]

    if s == 0:
        return tuple(out)
    # Here p must be 1-12
    if s == 1 and p <= 12:
        out += ["-define", f"tiff:lerc-compression={s}"]
        out += ["-define", f"tiff:lerc-zip-quality={p}"]
        return tuple(out)

    if s == 2:  # p 1-22
        out += ["-define", f"tiff:lerc-compression={s}"]
        out += ["-define", f"tiff:lerc-zstd-level={p}"]
        return tuple(out)

    raise ValueError(f"Invalid combination of s={s} and p={p} for LERC in ImageMagick")


IMAGEMAGICK_LERC_CONFIGS = [
    _C(
        name=f"imagemagick_lerc_effort_z{c['z']}_s{c['s']}_p{c['p']}",
        series="tiff-lerc",
        subcodec=_LIBTIFF_LERC_SUBCODECS[c["s"]],
        effort=c["p"],
        subcodec_index=c["s"],
        z_max_error=c["z"],
        allowed_dtypes=DTYPE_ANY,
        compressor=_im_comp(
            *_resolve_imagemagick_lerc(c["z"], c["s"], c["p"]),
        ),
        **_BC_LOSSLESS_IMAGEMAGICK,
    )
    for c in _LERC_COMBINATIONS
]


def _resolve_tifffile_lerc(z, s, p):
    compressionargs = {"level": z}
    if s != 0:
        compressionargs |= {
            "compression": _LIBTIFF_LERC_SUBCODECS[s],
            "compressionargs": {"level": p},
        }
    return {"compressionargs": compressionargs}


TIFFFILE_LERC_CONFIGS = [
    _C(
        name=f"tifffile_lerc_effort_z{c['z']}_s{c['s']}_p{c['p']}",
        series="tiff-lerc",
        subcodec=_LIBTIFF_LERC_SUBCODECS[c["s"]],
        effort=c["p"],
        subcodec_index=c["s"],
        z_max_error=c["z"],
        allowed_dtypes=DTYPE_ANY,
        compressor=(
            tifffile.compress_bulk,
            ("lerc",),
            _resolve_tifffile_lerc(c["z"], c["s"], c["p"]),
        ),
        **_BC_LOSSLESS_TIFFFILE,
    )
    for c in _LERC_COMBINATIONS
]
LERC_CONFIGS = LIBTIFF_LERC_CONFIGS + IMAGEMAGICK_LERC_CONFIGS + TIFFFILE_LERC_CONFIGS

# --- RLE ---
RLE_CONFIGS = [
    _C(
        name="libtiff_rle",
        series="tiff-rle",
        allowed_dtypes=DTYPE_ANY,
        compressor=_lt_comp("-c", "packbits"),
        **_BC_LOSSLESS_LIBTIFF,
    ),
    _C(
        name="imagemagick_rle",
        series="tiff-rle",
        allowed_dtypes=DTYPE_ANY,
        compressor=_im_comp("-compress", "RLE"),
        **_BC_LOSSLESS_IMAGEMAGICK,
    ),
    _C(
        name="tifffile_rle",
        series="tiff-rle",
        allowed_dtypes=DTYPE_ANY,
        compressor=(tifffile.compress_bulk, ("packbits",), {}),
        **_BC_LOSSLESS_TIFFFILE,
    ),
]


# Codec-specific combinations built from the shared dtype constants.
DTYPE_JPEG2000 = (
    DTYPE_UINT8_UINT16
    | DTYPE_UINT32
    | {
        np.int8,
        np.int16,
        np.int32,
    }
)
DTYPE_JPEGXL = {np.uint8, np.uint16, np.float16, np.float32}
DTYPE_JPEGXR = {np.uint8, np.uint16, np.float16, np.float32}
DTYPE_JPEGXR_LOSSLESS = {np.uint8, np.uint16, np.float16}
DTYPE_LERC = (
    DTYPE_UINT8_UINT16
    | DTYPE_UINT32
    | {np.int8, np.int16, np.int32, np.float32, np.float64}
)
DTYPE_LERC_IMAGEMAGICK = DTYPE_UINT8_UINT16 | {np.float32, np.float64}
DTYPE_SGILOG = {np.float32}


# --- Lossy LERC ---
_LERC_LOSSY_TOLERANCES = [0.5, 1.0, 2.0, 4.0, 8.0]
_LERC_LOSSY_SUBCODECS = [
    (0, "none", None),
    (1, "deflate", 6),
    (2, "zstd", 9),
]
_LERC_LOSSY_COMBINATIONS = [
    {"z": tolerance, "s": subcodec_index, "p": effort}
    for tolerance in _LERC_LOSSY_TOLERANCES
    for subcodec_index, _, effort in _LERC_LOSSY_SUBCODECS
]


def _lerc_lossy_name(implementation, combination):
    tolerance = f"{combination['z']:g}".replace(".", "p")
    effort = "" if combination["p"] is None else f"_p{combination['p']}"
    return f"{implementation}_lerc_lossy_z{tolerance}_s{combination['s']}{effort}"


def _resolve_libtiff_lerc(z, s, p):
    argument = f"lerc:{z:g}"
    if s != 0:
        argument += f":s{s}:p{p}"
    return _lt_comp("-c", argument)


LIBTIFF_LERC_LOSSY_CONFIGS = [
    _C(
        name=_lerc_lossy_name("libtiff", c),
        series="tiff-lerc",
        subseries="tiff-lerc-lossy-libtiff",
        lossless=False,
        tolerance=c["z"],
        effort=c["p"],
        subcodec=_LIBTIFF_LERC_SUBCODECS[c["s"]],
        subcodec_index=c["s"],
        allowed_dtypes=DTYPE_LERC,
        compressor=_resolve_libtiff_lerc(c["z"], c["s"], c["p"]),
        **_BC_LIBTIFF,
    )
    for c in _LERC_LOSSY_COMBINATIONS
]

IMAGEMAGICK_LERC_LOSSY_CONFIGS = [
    _C(
        name=_lerc_lossy_name("imagemagick", c),
        series="tiff-lerc",
        subseries="tiff-lerc-lossy-imagemagick",
        lossless=False,
        tolerance=c["z"],
        effort=c["p"],
        subcodec=_LIBTIFF_LERC_SUBCODECS[c["s"]],
        subcodec_index=c["s"],
        allowed_dtypes=DTYPE_LERC_IMAGEMAGICK,
        compressor=_im_comp(
            *_resolve_imagemagick_lerc(c["z"], c["s"], c["p"]),
        ),
        **_BC_IMAGEMAGICK,
    )
    for c in _LERC_LOSSY_COMBINATIONS
]

TIFFFILE_LERC_LOSSY_CONFIGS = [
    _C(
        name=_lerc_lossy_name("tifffile", c),
        series="tiff-lerc",
        subseries="tiff-lerc-lossy-tifffile",
        lossless=False,
        tolerance=c["z"],
        effort=c["p"],
        subcodec=_LIBTIFF_LERC_SUBCODECS[c["s"]],
        subcodec_index=c["s"],
        allowed_dtypes=DTYPE_LERC,
        compressor=(
            tifffile.compress_bulk,
            ("lerc",),
            _resolve_tifffile_lerc(c["z"], c["s"], c["p"]),
        ),
        **_BC_TIFFFILE,
    )
    for c in _LERC_LOSSY_COMBINATIONS
]

LERC_LOSSLESS_CONFIGS = LERC_CONFIGS
LERC_LOSSY_CONFIGS = (
    LIBTIFF_LERC_LOSSY_CONFIGS
    + IMAGEMAGICK_LERC_LOSSY_CONFIGS
    + TIFFFILE_LERC_LOSSY_CONFIGS
)
LERC_CONFIGS = LERC_LOSSLESS_CONFIGS + LERC_LOSSY_CONFIGS


# --- CCITT bilevel encodings (libtiff) ---
LIBTIFF_CCITT_CONFIGS = [
    _C(
        name=f"libtiff_ccitt_{name}",
        series="tiff-ccitt",
        subseries=f"tiff-ccitt-{name}",
        lossless=True,
        allowed_dtypes=DTYPE_BOOL,
        compressor=_lt_comp("-c", argument),
        **_BC_LIBTIFF,
    )
    for name, argument in (
        ("g3_1d", "g3:1d"),
        ("g3_2d", "g3:2d"),
        ("g3_2d_fill", "g3:2d:fill"),
        ("g4", "g4"),
    )
]


# --- JPEG (libtiff and tifffile/imagecodecs) ---
_JPEG_QUALITIES = [25, 50, 75, 90, 100]

LIBTIFF_JPEG_CONFIGS = [
    _C(
        name=f"libtiff_jpeg_q{quality}",
        series="tiff-jpeg",
        subseries="tiff-jpeg-lossy",
        lossless=False,
        quality=quality,
        allowed_dtypes=DTYPE_UINT8,
        compressor=_lt_comp("-c", f"jpeg:{quality}"),
        **_BC_LIBTIFF,
    )
    for quality in _JPEG_QUALITIES
]

TIFFFILE_JPEG_CONFIGS = [
    _C(
        name=f"tifffile_jpeg_q{quality}",
        series="tiff-jpeg",
        subseries="tiff-jpeg-lossy",
        lossless=False,
        quality=quality,
        allowed_dtypes=DTYPE_UINT8,
        compressor=_tf_comp("jpeg", compressionargs={"level": quality}),
        **_BC_TIFFFILE,
    )
    for quality in _JPEG_QUALITIES
]

# imagecodecs exposes JPEG lossless mode and predictors 1 through 7.  This is
# distinct from TIFF's top-level horizontal/floating-point predictor.
TIFFFILE_JPEG_LOSSLESS_CONFIGS = [
    _C(
        name=f"tifffile_jpeg_lossless_pred{predictor}",
        series="tiff-jpeg",
        subseries="tiff-jpeg-lossless",
        lossless=True,
        predictor=predictor,
        allowed_dtypes=(DTYPE_UINT8_UINT16 if predictor <= 4 else DTYPE_UINT8),
        compressor=_tf_comp(
            "jpeg",
            compressionargs={"lossless": True, "predictor": predictor},
        ),
        **_BC_TIFFFILE,
    )
    for predictor in range(1, 8)
]

JPEG_CONFIGS = (
    LIBTIFF_JPEG_CONFIGS + TIFFFILE_JPEG_CONFIGS + TIFFFILE_JPEG_LOSSLESS_CONFIGS
)


# --- JPEG 2000 (tifffile/imagecodecs) ---


TIFFFILE_JPEG2000_LOSSLESS_CONFIGS = [
    _C(
        name="tifffile_jpeg2000_lossless",
        series="tiff-jpeg2000",
        subseries="tiff-jpeg2000-lossless",
        lossless=True,
        allowed_dtypes=DTYPE_JPEG2000,
        compressor=_tf_comp("jpeg2000", compressionargs={"reversible": True}),
        **_BC_TIFFFILE,
    )
]

TIFFFILE_JPEG2000_LOSSY_CONFIGS = [
    _C(
        name=f"tifffile_jpeg2000_q{quality}",
        series="tiff-jpeg2000",
        subseries="tiff-jpeg2000-lossy",
        lossless=False,
        quality=quality,
        allowed_dtypes=DTYPE_JPEG2000,
        compressor=_tf_comp(
            "jpeg2000",
            compressionargs={"level": quality, "reversible": False},
        ),
        **_BC_TIFFFILE,
    )
    for quality in _JPEG2000_QUALITIES
]

TIFFFILE_JPEG2000_CONFIGS = (
    TIFFFILE_JPEG2000_LOSSLESS_CONFIGS + TIFFFILE_JPEG2000_LOSSY_CONFIGS
)


# --- JPEG XL (tifffile/imagecodecs) ---
_JPEGXL_EFFORTS = [1, 3, 5, 7, 9]
_JPEGXL_DISTANCES = [0.5, 1.0, 2.0, 4.0]

TIFFFILE_JPEGXL_LOSSLESS_CONFIGS = [
    _C(
        name=f"tifffile_jpegxl_lossless_e{effort}",
        series="tiff-jpegxl",
        subseries="tiff-jpegxl-lossless",
        lossless=True,
        effort=effort,
        allowed_dtypes=DTYPE_JPEGXL,
        compressor=_tf_comp(
            "jpegxl",
            compressionargs={"lossless": True, "effort": effort},
        ),
        **_BC_TIFFFILE,
    )
    for effort in _JPEGXL_EFFORTS
]

TIFFFILE_JPEGXL_LOSSY_CONFIGS = [
    _C(
        name=f"tifffile_jpegxl_d{distance:g}_e{effort}",
        series="tiff-jpegxl",
        subseries="tiff-jpegxl-lossy",
        lossless=False,
        effort=effort,
        allowed_dtypes=DTYPE_JPEGXL,
        extra={"distance": distance},
        compressor=_tf_comp(
            "jpegxl",
            compressionargs={"distance": distance, "effort": effort},
        ),
        **_BC_TIFFFILE,
    )
    for distance in _JPEGXL_DISTANCES
    for effort in _JPEGXL_EFFORTS
]

TIFFFILE_JPEGXL_CONFIGS = (
    TIFFFILE_JPEGXL_LOSSLESS_CONFIGS + TIFFFILE_JPEGXL_LOSSY_CONFIGS
)


# --- JPEG XR (tifffile/imagecodecs) ---
# Level 0 selects lossless coding.  Float32 is excluded from the lossless entry
# because the JPEG XR round trip quantizes it slightly; float16 is exact.
TIFFFILE_JPEGXR_LOSSLESS_CONFIGS = [
    _C(
        name="tifffile_jpegxr_lossless",
        series="tiff-jpegxr",
        subseries="tiff-jpegxr-lossless",
        lossless=True,
        quality=100,
        allowed_dtypes=DTYPE_JPEGXR_LOSSLESS,
        compressor=_tf_comp("jpegxr", compressionargs={"level": 0}),
        **_BC_TIFFFILE,
    )
]

TIFFFILE_JPEGXR_LOSSY_CONFIGS = [
    _C(
        name=f"tifffile_jpegxr_q{quality}",
        series="tiff-jpegxr",
        subseries="tiff-jpegxr-lossy",
        lossless=False,
        quality=quality,
        allowed_dtypes=DTYPE_JPEGXR,
        compressor=_tf_comp("jpegxr", compressionargs={"level": quality / 100}),
        **_BC_TIFFFILE,
    )
    for quality in _JPEG_XR_QUALITIES
]

TIFFFILE_JPEGXR_CONFIGS = (
    TIFFFILE_JPEGXR_LOSSLESS_CONFIGS + TIFFFILE_JPEGXR_LOSSY_CONFIGS
)


# --- PNG-in-TIFF (tifffile/imagecodecs) ---
TIFFFILE_PNG_CONFIGS = [
    _C(
        name=f"tifffile_png_p{effort}",
        series="tiff-png",
        lossless=True,
        effort=effort,
        allowed_dtypes=DTYPE_UINT8_UINT16,
        compressor=_tf_comp("png", compressionargs={"level": effort}),
        **_BC_TIFFFILE,
    )
    for effort in _PNG_EFFORTS
]


# --- SGILog (libtiff) ---
# SGILog stores floating-point luminance in a logarithmic integer
# representation.  It is therefore lossy and is only useful here for float32
# input.  Replace the overly broad inherited configs.py definition in CONFIGS.
LIBTIFF_SGILOG_CONFIGS = [
    _C(
        name="libtiff_sgilog",
        series="tiff-sgilog",
        lossless=False,
        allowed_dtypes=DTYPE_SGILOG,
        compressor=_lt_comp("-c", "sgilog"),
        **_BC_LIBTIFF,
    )
]

CONFIGS = (
    []
    + BASELINE_CONFIGS
    + LZW_CONFIGS
    + DEFLATE_CONFIGS
    + LZMA_CONFIGS
    + ZSTD_CONFIGS
    + LERC_LOSSLESS_CONFIGS
    + LIBTIFF_SGILOG_CONFIGS
    + RLE_CONFIGS
    + LERC_LOSSY_CONFIGS
    + LIBTIFF_CCITT_CONFIGS
    + JPEG_CONFIGS
    + TIFFFILE_JPEG2000_CONFIGS
    + TIFFFILE_JPEGXL_CONFIGS
    + TIFFFILE_JPEGXR_CONFIGS
    + TIFFFILE_PNG_CONFIGS
)

CONFIGS_LOSSY = [c for c in CONFIGS if not c.lossless]
CONFIGS_LOSSLESS = [c for c in CONFIGS if c.lossless]

In [11]:
# Check duplicate names, happens if we forgot f in front of f-string
names = [c.name for c in CONFIGS]
if len(names) != len(set(names)):
    duplicates = set([name for name in names if names.count(name) > 1])
    raise ValueError(f"Duplicate config names found: {duplicates}")


In [ ]:
BASE_EXTRA = {
    "bulk_runner_kwargs": {"n_jobs": 16, "joblib_kwargs": {"prefer": "processes"}},
}

if target_configs is not None:
    TARGET_CONFIGS = globals()[target_configs]
else:
    # TARGET_CONFIGS = CONFIGS
    TARGET_CONFIGS = CONFIGS_LOSSLESS

_reverser = -1 if reverse_configs else 1
TARGET_CONFIGS = TARGET_CONFIGS[::_reverser]

initial_size = sum(f.stat().st_size for f in INPUT_FILES)
tests = []

logger.info(
    "Number of configs: %d, lossy: %d, lossless: %d, reversed: %s",
    len(TARGET_CONFIGS),
    len([c for c in TARGET_CONFIGS if not c.lossless]),
    len([c for c in TARGET_CONFIGS if c.lossless]),
    reverse_configs,
)
logger.info("Input files: %d", len(INPUT_FILES))
logger.info("Initial input size: %d bytes", initial_size)


for config in TARGET_CONFIGS:
    try:
        if INPUT_DTYPE not in config.allowed_dtypes:
            logger.warning(
                "Skipping config %s because dtype %s is unsupported",
                config.name,
                INPUT_DTYPE,
            )
            continue

        logger.info("Starting config: %s", config.name)

        start_wall_dt = datetime.now(UTC)
        total_wall_time_start = perf_counter()
        extra = BASE_EXTRA.copy()

        # Put in tmpfs to avoid disk I/O bottleneck
        # This is automatically deleted at end of scope/context manager
        with mem_temper.TemporaryDirectory() as compress_temp_dir:
            logger.debug(
                "Created temporary directory for compression: %s",
                compress_temp_dir,
            )
            compress_temp_dir_path = Path(compress_temp_dir)

            # Run compression
            compress_start = perf_counter()
            compressed_files = [
                compress_temp_dir_path / (input_file.name) for input_file in INPUT_FILES
            ]
            logger.info("Compressing with config %s", config.name)
            compression_results = config.compressor[0](
                INPUT_FILES,
                compressed_files,
                *config.compressor[1],
                **config.compressor[2],
                **extra["bulk_runner_kwargs"],
            )
            compress_wall_time_seconds = perf_counter() - compress_start
            logger.info(
                "Compression completed for %s in %.3f seconds",
                config.name,
                compress_wall_time_seconds,
            )
            compress_user_time_seconds = sum(r.user_time for r in compression_results)
            compress_system_time_seconds = sum(
                r.system_time for r in compression_results
            )

            # Run decompression
            with mem_temper.TemporaryDirectory() as decompress_temp_dir:
                logger.debug(
                    "Created temporary directory for decompression: %s",
                    decompress_temp_dir,
                )
                decompress_temp_dir_path = Path(decompress_temp_dir)
                decompressed_files = [
                    decompress_temp_dir_path / (input_file.name)
                    for input_file in INPUT_FILES
                ]
                decompress_start = perf_counter()
                logger.info("Decompressing output from config %s", config.name)
                decompression_results = config.decompressor[0](
                    compressed_files,
                    decompressed_files,
                    *config.decompressor[1],
                    **config.decompressor[2],
                    **extra["bulk_runner_kwargs"],
                )
                decompress_wall_time_seconds = perf_counter() - decompress_start
                logger.info(
                    "Decompression completed for %s in %.3f seconds",
                    config.name,
                    decompress_wall_time_seconds,
                )
                decompress_user_time_seconds = sum(
                    r.user_time for r in decompression_results
                )
                decompress_system_time_seconds = sum(
                    r.system_time for r in decompression_results
                )

                # Calculate size before releasing from memory
                uncompressed_size = initial_size

                # Get individual file sizes
                images_input_sizes = {}
                images_uncompressed_sizes = {}
                images_compressed_sizes = {}
                for input_file, compressed_file, decompressed_file in zip(
                    INPUT_FILES, compressed_files, decompressed_files
                ):
                    images_input_sizes[input_file.name] = input_file.stat().st_size
                    images_uncompressed_sizes[input_file.name] = (
                        decompressed_file.stat().st_size
                    )
                    images_compressed_sizes[input_file.name] = (
                        compressed_file.stat().st_size
                    )

                assert sum(images_input_sizes.values()) == initial_size
                actual_uncompressed_size = sum(images_uncompressed_sizes.values())
                compressed_size = sum(images_compressed_sizes.values())

                # Run comparison
                logger.info("Comparing reconstructed files for config %s", config.name)
                comparisons = config.comparer[0](
                    INPUT_FILES,
                    decompressed_files,
                    *config.comparer[1],
                    **config.comparer[2],
                    **extra["bulk_runner_kwargs"],
                )

                was_actually_lossless = all(comparisons)
                logger.info(
                    "Comparison completed for %s: lossless=%s",
                    config.name,
                    was_actually_lossless,
                )

        # Get pixel count for bitrate
        n_elements = 0
        for input_file in INPUT_FILES:
            # Note that this does not actually read image data, so not super slow
            with TiffFile(input_file) as tif:
                for page in tif.pages:
                    n_elements += prod(page.shape)

        # Calculate quantities
        compression_wall_rate = uncompressed_size / compress_wall_time_seconds
        compression_user_rate = uncompressed_size / compress_user_time_seconds
        decompression_wall_rate = uncompressed_size / decompress_wall_time_seconds
        decompression_user_rate = uncompressed_size / decompress_user_time_seconds

        bitrate_uncompressed = 8 * initial_size / n_elements
        bitrate_compressed = 8 * compressed_size / n_elements

        compression_ratio = uncompressed_size / compressed_size
        space_saving = 1 - 1 / compression_ratio

        end_wall_dt = datetime.now(UTC)
        total_wall_time_seconds = perf_counter() - total_wall_time_start

        test = CompressionTest(
            **config.model_dump(),
            id=config.get_pathlike_str(),
            dataset_name=DATASET,
            dataset_path=DATASET_DIR,
            start_wall_dt=start_wall_dt,
            end_wall_dt=end_wall_dt,
            total_wall_time_seconds=total_wall_time_seconds,
            config=config,
            baseline_test=None,  # TODO: For now
            host=host,
            compress_wall_time_seconds=compress_wall_time_seconds,
            compress_user_time_seconds=compress_user_time_seconds,
            compress_system_time_seconds=compress_system_time_seconds,
            decompress_wall_time_seconds=decompress_wall_time_seconds,
            decompress_user_time_seconds=decompress_user_time_seconds,
            decompress_system_time_seconds=decompress_system_time_seconds,
            uncompressed_size=uncompressed_size,
            actual_uncompressed_size=actual_uncompressed_size,
            compressed_size=compressed_size,
            n_elements=n_elements,
            compression_wall_rate=compression_wall_rate,
            compression_user_rate=compression_user_rate,
            decompression_wall_rate=decompression_wall_rate,
            decompression_user_rate=decompression_user_rate,
            bitrate_uncompressed=bitrate_uncompressed,
            bitrate_compressed=bitrate_compressed,
            compression_ratio=compression_ratio,
            space_saving=space_saving,
            was_actually_lossless=was_actually_lossless,
            images_input_sizes=images_input_sizes,
            images_uncompressed_sizes=images_uncompressed_sizes,
            images_compressed_sizes=images_compressed_sizes,
            runner_extra=extra,
        )

        tests.append(test)

        logger.debug("Test result:\n%s", pformat(test.summary(), sort_dicts=False))

        output_file = TEST_OUTPUT_DIR / f"test_{test.name}.json"
        with open(output_file, "w") as f:
            f.write(test.model_dump_json(indent=4, fallback=repr))

        logger.info(
            "Completed config %s: ratio=%.4f, space_saving=%.2f%%, result=%s",
            config.name,
            compression_ratio,
            space_saving * 100,
            output_file,
        )

    except Exception:
        logger.exception("Error running test for config %s", config.name)

2026-08-02 14:08:33 | INFO     | Number of configs: 337, lossy: 0, lossless: 337, reversed: False
2026-08-02 14:08:33 | INFO     | Input files: 1572
2026-08-02 14:08:33 | INFO     | Initial input size: 12576198072 bytes
2026-08-02 14:08:33 | INFO     | Starting config: libtiff_baseline
2026-08-02 14:08:33 | DEBUG    | Created temporary directory for compression: /dev/shm/tmps5sy8bh4
2026-08-02 14:08:33 | INFO     | Compressing with config libtiff_baseline
2026-08-02 14:08:42 | INFO     | Compression completed for libtiff_baseline in 9.124 seconds
2026-08-02 14:08:42 | DEBUG    | Created temporary directory for decompression: /dev/shm/tmp86noqn0w
2026-08-02 14:08:42 | INFO     | Decompressing output from config libtiff_baseline
2026-08-02 14:08:50 | INFO     | Decompression completed for libtiff_baseline in 8.304 seconds
2026-08-02 14:08:50 | INFO     | Comparing reconstructed files for config libtiff_baseline
2026-08-02 14:09:04 | INFO     | Comparison completed for libtiff_baseline: l

In [ ]:
# Clean up memory
CLEAR_MEMORY = True

if is_papermill or CLEAR_MEMORY:
    shutil.rmtree(temp_dir_path)